# SEED-VII EEGNet Encoder Pipeline

**重构版 (2026-05-27)：EEGNet 主力，无 ICA，per-subject .npz 预处理**

## 执行流程
1. **Cell 0**: 安装依赖
2. **Cell 1**: 环境配置 & 路径设置
3. **Cell 2**: 预处理 — 20 个 .mat → 20 个 .npz
4. **Cell 3**: 上传 .npz 到 ModelScope
5. **Cell 4**: (可选) 从 ModelScope 下载 .npz
6. **Cell 5**: 训练 — 全被试混合，一个模型（原始行为）
7. **Cell 6**: 训练 — 每个被试独立训练一个模型（被试内泛化）
8. **Cell 7**: 续训（全被试混合模式）
9. **Cell 8**: 编码器推理
10. **Cell 9**: 查看训练结果

---

## 两种训练模式说明

| 模式 | 脚本 | 输出 | 适用场景 |
|------|------|------|----------|
| 全被试混合 | `train.py` | 1 个模型 | 跨被试泛化研究 |
| 每被试独立 | `train_per_subject.py` | N 个模型（每被试一个） | 被试内泛化，EEG 跨被试泛化差是公认结论，此模式更合理 |

**两种模式均严格 trial-level 分割**：同一 trial 的所有时间窗口只出现在 train/val/test 之一中，无数据泄漏。

---

## Cell 0: 安装依赖

In [ ]:
!pip install -q numpy scipy torch tqdm h5py pyyaml modelscope

## Cell 1: 环境配置 & 路径设置

**请根据你的实际环境修改以下路径变量！**

In [ ]:
import os, sys
from pathlib import Path

# ========== 路径配置 ==========
PROJECT_ROOT  = Path("/mnt/workspace/EEG_encoder_version2")  # 修改为实际路径
MAT_DIR       = Path("/mnt/workspace/data/EEG_preprocessed")
SAVE_INFO_DIR = Path("/mnt/workspace/data/save_info")
NPZ_DIR       = Path("/mnt/workspace/preprocessed")

# 全被试混合训练输出目录
RUNS_DIR = Path("/mnt/workspace/runs")

# 单被试独立训练输出目录
RUNS_PER_SUBJ_DIR = Path("/mnt/workspace/runs_per_subject")

# 编码输出
ENCODED_OUTPUT = Path("/mnt/workspace/encoded.npz")

# ModelScope
MODELSCOPE_DATASET = "DEREKVERSE/SEED-VII"
NPZ_REPO_PREFIX    = "preprocessed_npz"
# os.environ["MODELSCOPE_API_TOKEN"] = "your_token_here"

sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT:      {PROJECT_ROOT} (exists: {PROJECT_ROOT.exists()})")
print(f"MAT_DIR:           {MAT_DIR} (exists: {MAT_DIR.exists()})")
print(f"NPZ_DIR:           {NPZ_DIR}")
print(f"RUNS_DIR:          {RUNS_DIR}")
print(f"RUNS_PER_SUBJ_DIR: {RUNS_PER_SUBJ_DIR}")

## Cell 2: 预处理 — 20 个 .mat → 20 个 per-subject .npz

```
原始 EEG (62通道, 200Hz)
    ↓ 基线校正 → CAR → 居中60%裁剪 → 4s窗口50%重叠 → z-score
    → {subject}.npz
```

In [ ]:
!python {PROJECT_ROOT}/scripts/preprocess_to_npz.py \
  --mat-dir {MAT_DIR} \
  --output-dir {NPZ_DIR} \
  --save-info-dir {SAVE_INFO_DIR if SAVE_INFO_DIR.exists() else ''} \
  --window-seconds 4.0 \
  --step-seconds 2.0 \
  --middle-ratio 0.6 \
  --max-windows-per-trial 60 \
  --compress

In [ ]:
# 验证预处理结果
import numpy as np
for p in sorted(NPZ_DIR.glob("*.npz")):
    data = np.load(p, allow_pickle=True)
    print(f"{p.name}: X={data['X'].shape}, y={data['y'].shape}, "
          f"labels={dict(zip(*np.unique(data['y'], return_counts=True)))}")

## Cell 3: 上传 .npz 到 ModelScope

In [ ]:
!python {PROJECT_ROOT}/scripts/upload_npz_to_modelscope.py \
  --npz-dir {NPZ_DIR} \
  --dataset {MODELSCOPE_DATASET} \
  --path-prefix {NPZ_REPO_PREFIX}

## Cell 4: (可选) 从 ModelScope 下载 .npz

In [ ]:
# !python {PROJECT_ROOT}/scripts/download_npz_from_modelscope.py \
#   --dataset {MODELSCOPE_DATASET} \
#   --path-prefix {NPZ_REPO_PREFIX} \
#   --local-dir {NPZ_DIR}

## Cell 5: 训练 — 全被试混合，一个模型（原始行为）

所有 20 个被试的数据混合后做 trial-level 分割，训练一个共享模型。  
适合研究跨被试泛化（但 EEG 跨被试泛化能力公认较差）。

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_DIR} \
  --model-type eegnet \
  --device auto \
  --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

## Cell 6: 训练 — 每个被试独立训练一个模型

**这是被试内泛化的正确做法。**

### 逻辑
对每个被试 S（循环 20 次）：
1. 只加载 `S.npz`（该被试自己的数据）
2. 在 **S 自己的 trial** 内做 trial-level 分割：
   - train：S 的 80% trial 的全部时间窗口
   - val：  S 的 10% trial 的全部时间窗口（用于早停 & 选最优模型）
   - test： S 的 10% trial 的全部时间窗口（最终评估，训练中不可见）
3. 从头初始化一个新模型，完整跑训练循环（预训练→联合训练→早停）
4. 保存到 `runs_per_subject/subject_S/`
5. 释放该模型和显存，进入下一个被试

**数据泄漏防护**：同一 trial 的所有时间窗口整体地属于 train/val/test 之一，绝不跨界。

### 输出结构
```
runs_per_subject/
  subject_1/
    best_encoder.pt   ← 被试 1 的编码器
    best_model.pt
    summary.json      ← 被试 1 的 val_acc / test_acc / test_mae
    train.log
  subject_2/ ...
  ...
  subject_20/ ...
  all_summary.json    ← 汇总：20 个被试的均值±标准差
```

In [ ]:
!python {PROJECT_ROOT}/scripts/train_per_subject.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_PER_SUBJ_DIR} \
  --model-type eegnet \
  --device auto \
  --amp \
  --batch-size 64 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --max-epochs 200 \
  --patience 30 \
  --val-ratio 0.1 \
  --test-ratio 0.1 \
  --max-runtime-hours 20

In [ ]:
# 仅跑部分被试（调试/验证用）
# !python {PROJECT_ROOT}/scripts/train_per_subject.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RUNS_PER_SUBJ_DIR} \
#   --subjects 1,2,3 \
#   --model-type eegnet \
#   --device auto --amp \
#   --max-epochs 50 \
#   --max-runtime-hours 3

In [ ]:
# 查看所有被试的汇总结果
import json
import numpy as np

all_summary_path = RUNS_PER_SUBJ_DIR / "all_summary.json"
if all_summary_path.exists():
    with open(all_summary_path) as f:
        all_summary = json.load(f)

    print(f"训练被试数: {all_summary['n_subjects']}")
    print(f"Val  Acc:  mean={all_summary['mean_val_acc']:.4f} ± {all_summary['std_val_acc']:.4f}")
    if all_summary.get('mean_test_acc') is not None:
        print(f"Test Acc:  mean={all_summary['mean_test_acc']:.4f} ± {all_summary['std_test_acc']:.4f}")
        print(f"Test MAE:  mean={all_summary['mean_test_mae']:.4f} ± {all_summary['std_test_mae']:.4f}")

    print("\n各被试明细:")
    print(f"  {'Subject':>10} | {'Val Acc':>8} | {'Test Acc':>9} | {'Test MAE':>9} | {'Epochs':>6}")
    print("  " + "-"*55)
    for subj, sm in sorted(all_summary["per_subject"].items(), key=lambda x: x[0]):
        test_acc = sm.get("test", {}).get("acc", float("nan"))
        test_mae = sm.get("test", {}).get("intensity_mae", float("nan"))
        print(f"  {subj:>10} | {sm['best_val_acc']:>8.4f} | {test_acc:>9.4f} | "
              f"{test_mae:>9.4f} | {sm['epochs_run']:>6}")
else:
    print(f"all_summary.json 不存在，请先运行 Cell 6 的训练。")

## Cell 7: 续训（全被试混合模式）

In [ ]:
# !python {PROJECT_ROOT}/scripts/train.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RUNS_DIR} \
#   --model-type eegnet \
#   --device auto --amp \
#   --resume \
#   --max-runtime-hours 10

## Cell 8: 编码器推理

- **全被试混合模型**：用单个 `best_encoder.pt` 对所有数据编码
- **单被试模型**：用每个被试自己的 `best_encoder.pt` 编码自己的数据

In [ ]:
# 全被试混合模型推理
!python {PROJECT_ROOT}/scripts/encode.py \
  --data-dir {NPZ_DIR} \
  --checkpoint {RUNS_DIR}/best_encoder.pt \
  --output {ENCODED_OUTPUT} \
  --model-type eegnet \
  --device auto

In [ ]:
# 单被试模型推理：每个被试用自己的编码器
import numpy as np, json
from pathlib import Path
import subprocess, sys

per_subj_encoded_dir = RUNS_PER_SUBJ_DIR / "encoded"
per_subj_encoded_dir.mkdir(exist_ok=True)

for subj_dir in sorted(RUNS_PER_SUBJ_DIR.glob("subject_*")):
    subj = subj_dir.name.replace("subject_", "")
    ckpt = subj_dir / "best_encoder.pt"
    if not ckpt.exists():
        print(f"[Subject {subj}] best_encoder.pt 不存在，跳过")
        continue
    out = per_subj_encoded_dir / f"encoded_{subj}.npz"
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "encode.py"),
        "--data-dir",   str(NPZ_DIR),
        "--checkpoint", str(ckpt),
        "--output",     str(out),
        "--subjects",   subj,
        "--model-type", "eegnet",
        "--device",     "auto",
    ]
    print(f"[Subject {subj}] Encoding...")
    subprocess.run(cmd, check=True)
    data = np.load(str(out), allow_pickle=True)
    acc  = (data["cls_pred"] == data["labels"]).mean()
    print(f"[Subject {subj}] features={data['features'].shape}, acc={acc:.4f}")

## Cell 9: 查看训练日志

In [ ]:
import json

# 全被试混合模型摘要
summary_path = RUNS_DIR / "summary.json"
if summary_path.exists():
    print("=== 全被试混合模型 ===")
    with open(summary_path) as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# 单被试模型汇总
all_summary_path = RUNS_PER_SUBJ_DIR / "all_summary.json"
if all_summary_path.exists():
    print("\n=== 单被试独立模型汇总 ===")
    with open(all_summary_path) as f:
        s = json.load(f)
    # 只打印聚合部分，不打印 per_subject 详情（太长）
    print(json.dumps({k: v for k, v in s.items() if k != "per_subject"},
                     indent=2, ensure_ascii=False))

In [ ]:
# 查看某个被试的详细训练日志
SUBJECT_TO_INSPECT = "1"   # 修改为你想看的被试编号

log_path = RUNS_PER_SUBJ_DIR / f"subject_{SUBJECT_TO_INSPECT}" / "train.log"
if log_path.exists():
    with open(log_path) as f:
        lines = f.readlines()
    print(f"--- Subject {SUBJECT_TO_INSPECT} 最后 30 行 ---")
    for line in lines[-30:]:
        print(line, end="")
else:
    print(f"{log_path} 不存在")